## React Agent
Here we can use the Python SDK to develop the simple react agent, then save the agent to a config.yaml and run it from there.

In [1]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)

In [2]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

In [3]:
from nat.agent.react_agent.register import NatReActAgent
from nat.llm.nim_llm import NimLLM
from nat.plugins.langchain.tools.code_generation_tool import CodeGenerationTool
from nat.plugins.langchain.tools.wikipedia_search import WikiSearchTool
from nat.tool.datetime_tools import CurrentTimeTool
from nat.utils.sdk.nat_workflow import NatWorkflow

llm = NimLLM(
    model_name="nvdev/meta/llama-3.3-70b-instruct",
    temperature=0,
    max_tokens=250,
    name="nim_llm",
)

wikipedia_search_tool = WikiSearchTool(
    max_results=3,
    name="wiki_search",
)

current_time_tool = CurrentTimeTool(
    name="current_datetime",
)

generate_code_tool = CodeGenerationTool(
    programming_language="Python",
    description="Useful to generate Python code. For any questions about code generation, you must only use this tool!",
    llm=llm,
    verbose=True,
    name="code_generation",
)

agent = NatReActAgent(
    tools=[wikipedia_search_tool, current_time_tool, generate_code_tool],
    llm=llm,
    verbose=True,
    parse_agent_response_max_retries=2,
)

nat_workflow = NatWorkflow(
    entrypoint=agent,
)

/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the agent to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


functions:
  wiki_search:
    _type: wiki_search
    max_results: 3
  current_datetime:
    _type: current_datetime
  code_generation:
    _type: code_generation
    llm_name: nim_llm
    verbose: true
    programming_language: Python
    description: |-
      Useful to generate Python code. For any questions about code generation, you must only use this
      tool!

llms:
  nim_llm:
    _type: nim
    model: nvdev/meta/llama-3.3-70b-instruct
    max_tokens: 250
    temperature: 0.0

workflow:
  _type: react_agent
  llm_name: nim_llm
  verbose: true
  tool_names:
  - wiki_search
  - current_datetime
  - code_generation
  parse_agent_response_max_retries: 2



In [5]:
await nat_workflow.prompt("Who was Djikstra?")

Discovered frameworks: {<LLMFrameworkEnum.LANGCHAIN: 'langchain'>} in function code_generation_tool by inspecting source. It is recommended and more reliable to instead add the used LLMFrameworkEnum types in the framework_wrappers argument when calling @register_function.
[AGENT] ReAct Agent wants to call tool None. In the ReAct Agent's configuration within the config file,there is no tool with that name: ['wiki_search', 'current_datetime', 'code_generation']


"Edsger W. Dijkstra was a Dutch computer scientist, programmer, mathematician, and science essayist, best known for developing Dijkstra's algorithm, a method for finding the shortest path in a weighted graph. He made significant contributions to the field of computer science and was awarded the Turing Award in 1972 for his work on structured programming languages."

In [ ]:
from pathlib import Path

from nat.eval.rag_evaluator.register import RagasEvaluator
from nat.utils.sdk.nat_evaluation import EvalDatasetJsonConfig
from nat.utils.sdk.nat_evaluation import NatEvaluation

path_to_dataset = Path(os.path.curdir, "../../../../", "examples/agents/data/wikipedia.json").resolve()

accuracy_evaluator = RagasEvaluator(
    llm=llm,
    metric="AnswerAccuracy",
    name="accuracy"
)

relevance_evaluator = RagasEvaluator(
    llm=llm,
    metric="ContextRelevance",
    name="relevance"
)

response_groundedness_evaluator = RagasEvaluator(
    llm=llm,
    metric="ResponseGroundedness",
    name="groundedness"
)

# Create evaluation config directly - no need for separate NatGeneralEvaluator
evaluation = NatEvaluation(
    output_dir=Path(".tmp/nat/examples/react_agent/"),
    dataset=EvalDatasetJsonConfig(file_path=path_to_dataset),
    evaluators=[accuracy_evaluator, relevance_evaluator, response_groundedness_evaluator],
)

nat_workflow.add_evaluator(evaluation)

In [7]:
path_to_yaml = Path(os.getcwd(), "config", "eval_config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the agent to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())

functions:
  wiki_search:
    _type: wiki_search
    max_results: 3
  current_datetime:
    _type: current_datetime
  code_generation:
    _type: code_generation
    llm_name: nim_llm
    verbose: true
    programming_language: Python
    description: |-
      Useful to generate Python code. For any questions about code generation, you must only use this
      tool!

llms:
  nim_llm:
    _type: nim
    model: nvdev/meta/llama-3.3-70b-instruct
    max_tokens: 250
    temperature: 0.0

workflow:
  _type: react_agent
  llm_name: nim_llm
  verbose: true
  tool_names:
  - wiki_search
  - current_datetime
  - code_generation
  parse_agent_response_max_retries: 2

eval:
  general:
    output_dir: .tmp/nat/examples/react_agent
    dataset:
      _type: json
      file_path: /Users/spastoriza/Documents/Programming/public/nat-fork/examples/agents/data/wikipedia.json
  evaluators:
    accuracy:
      _type: ragas
      llm_name: nim_llm
      metric: AnswerAccuracy
    relevance:
      _type: rag

In [8]:
await nat_workflow.evaluate()

Discovered frameworks: {<LLMFrameworkEnum.LANGCHAIN: 'langchain'>} in function code_generation_tool by inspecting source. It is recommended and more reliable to instead add the used LLMFrameworkEnum types in the framework_wrappers argument when calling @register_function.
Evaluating Ragas nv_accuracy:   0%|          | 0/3 [00:00<?, ?it/s]

Evaluating Ragas nv_accuracy: 100%|██████████| 3/3 [00:04<00:00,  1.38s/it]
